# 05. 시각화

문화결핍지수 지도, 차트, 인터랙티브 지도를 생성합니다.

생성 파일:
- `outputs/figures/top_risk_regions.png`
- `outputs/figures/province_summary.png`
- `outputs/figures/score_distribution.png`
- `outputs/figures/choropleth_map.html`

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd, warnings
warnings.filterwarnings('ignore')
from preprocessing import load_raw_data
from feature_engineering import build_region_features_from_raw
from scoring import add_scores
from clustering import add_clusters
from recommendation import add_recommendations
from visualization import save_all_charts
from pathlib import Path

raw = load_raw_data()
features = build_region_features_from_raw(raw)
scored = add_recommendations(add_clusters(add_scores(features)))
print(f'대상 시군구: {len(scored)}개')

## 5-1. Top 15 차트

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

top = scored.sort_values('culture_gap_score', ascending=False).head(15)
colors = ['#c84c3a' if r=='주의' else '#e8956b' for r in top['risk_level']]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top['region_name'], top['culture_gap_score'], color=colors)
ax.invert_yaxis()
ax.axvline(x=60, color='red', linestyle='--', linewidth=0.8)
ax.set_xlabel('문화결핍지수')
ax.set_title('문화 사각지대 고위험 시군구 Top 15', fontweight='bold')
plt.tight_layout()
plt.show()

## 5-2. 시도별 요약 차트

In [ ]:
prov = scored.groupby('province')['culture_gap_score'].agg(['mean','count']).sort_values('mean', ascending=False)
risk = scored.groupby(['province','risk_level']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(prov.index, prov['mean'],
             color=['#c84c3a' if s>=60 else '#e8956b' for s in prov['mean']])
axes[0].set_title('시도별 평균 문화결핍지수')
axes[0].invert_yaxis()

risk_cols = [c for c in ['주의','보통','양호'] if c in risk.columns]
risk_colors = {'주의':'#e8956b','보통':'#f5c842','양호':'#4caf7d'}
bottom = pd.Series(0, index=risk.index)
for col in risk_cols:
    axes[1].barh(risk.index, risk[col], left=bottom, label=col, color=risk_colors[col])
    bottom += risk[col]
axes[1].legend()
axes[1].set_title('시도별 위험 등급 분포')
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 5-3. 인터랙티브 지도 생성 및 저장

In [ ]:
geojson_path = Path('../data/raw/sigungu_boundary_full.geojson')
figure_dir = Path('../outputs/figures')
save_all_charts(scored, figure_dir, geojson_path)
print('저장 완료:')
for f in sorted(figure_dir.iterdir()):
    print(f'  {f.name}')

## 5-4. 인터랙티브 지도 미리보기 (Jupyter)

In [ ]:
from IPython.display import IFrame
IFrame('../outputs/figures/choropleth_map.html', width=900, height=550)